<a href="https://colab.research.google.com/github/Hana-ElGabry/CareerPilot/blob/main/01_Data_Ingestion_and_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1. Import Libraries and Load Data

In [5]:
import pandas as pd
import duckdb
import os
import glob
import time
from pathlib import Path

In [6]:
trips_dataset_path = '/content/drive/MyDrive/NETMOB25/data/trips_dataset.csv'
individuals_dataset_path = '/content/drive/MyDrive/NETMOB25/data/individuals_dataset.csv'
gpr_dataset_path = '/content/drive/MyDrive/NETMOB25/data/gps_dataset'
osm_path = '/content/drive/MyDrive/NETMOB25/Copy of gis_osm_roads_free_1.shp'

In [7]:
individuals_df = pd.read_csv(individuals_dataset_path)
trips_df = pd.read_csv(trips_dataset_path)

In [12]:
individuals_df.head(3)

,ID,CODGEO,AREA_NAME,SEX,AGE,DIPLOMA,PRO_CAT,TYPE_HOUSE,NBPERS_HOUSE,NB_10,...,TWO_WHEELER,BIKE,ELECT_SCOOTER,NAVIGO_SUB,IMAGINER_SUB,OTHER_SUB_PT,BIKE_SUB,NSM_SUB,WEIGHT_INDIV,GPS_RECORD
0,10_2978,78092,Bougival,Woman,41,NaN,4,In a couple with child(ren),2,1,...,1,2,0,False,False,False,False,False,1856.206160,True
1,10_2980,75120,Paris,Man,30,5-year-and-above higher education degree: Mast...,2,Living alone,1,0,...,0,0,0,True,False,False,False,False,1375.000372,True
2,10_2981,91326,Juvisy-sur-Orge,Man,38,5-year-and-above higher education degree: Mast...,2,In a couple without children,2,0,...,0,1,0,True,False,False,False,False,1231.812990,True


In [11]:
trips_df.head(3)

,KEY,ID,Day_EMG,Date_EMG,Day_Type,ID_Trip_Days,Type_Trip_OD,Area_O,Area_D,Code_INSEE_O,...,Duration,Purpose_O,Purpose_D,Main_Mode,Mode_1,Mode_2,Mode_3,Mode_4,Mode_5,Weight_Day
0,42_0001-monday-44851-PDD,42_0001,monday,2022-10-17,Normal,No_Trip,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,97.171946
1,42_0001-tuesday-44852-1,42_0001,tuesday,2022-10-18,Strike,1,Internal,SAVIGNY-SUR-ORGE,SAINT-MAUR-DES-FOSSES,91589,...,72.0,DEPART_HOME,OTHER,PRIV_CAR_PASSENGER,PRIV_CAR_PASSENGER,NaN,NaN,NaN,NaN,92.090337
2,42_0001-tuesday-44852-2,42_0001,tuesday,2022-10-18,Strike,2,Internal,SAINT-MAUR-DES-FOSSES,SAINTE-GENEVIEVE-DES-BOIS,94068,...,39.0,OTHER,LEISURE,PRIV_CAR_PASSENGER,PRIV_CAR_PASSENGER,NaN,NaN,NaN,NaN,92.090337


### 2. Inspect Initial Data


In [13]:
individuals_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3337 entries, 0 to 3336
Data columns (total 27 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               3337 non-null   object 
 1   CODGEO           3337 non-null   int64  
 2   AREA_NAME        3337 non-null   object 
 3   SEX              3337 non-null   object 
 4   AGE              3337 non-null   int64  
 5   DIPLOMA          2811 non-null   object 
 6   PRO_CAT          3337 non-null   int64  
 7   TYPE_HOUSE       3337 non-null   object 
 8   NBPERS_HOUSE     3337 non-null   int64  
 9   NB_10            3337 non-null   int64  
 10  NB_11_17         3337 non-null   float64
 11  NB_18_24         3337 non-null   float64
 12  NB_25_64         3337 non-null   int64  
 13  NB_65            3337 non-null   int64  
 14  PMR              3337 non-null   bool   
 15  DRIVING_LICENCE  3322 non-null   object 
 16  NB_CAR           3337 non-null   int64  
 17  TWO_WHEELER   

In [14]:
trips_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80697 entries, 0 to 80696
Data columns (total 27 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   KEY           80697 non-null  object 
 1   ID            80697 non-null  object 
 2   Day_EMG       80697 non-null  object 
 3   Date_EMG      80697 non-null  object 
 4   Day_Type      80697 non-null  object 
 5   ID_Trip_Days  80697 non-null  object 
 6   Type_Trip_OD  77002 non-null  object 
 7   Area_O        76717 non-null  object 
 8   Area_D        76717 non-null  object 
 9   Code_INSEE_O  76717 non-null  object 
 10  Code_INSEE_D  76717 non-null  object 
 11  Zone_O        76301 non-null  object 
 12  Zone_D        76203 non-null  object 
 13  Date_O        76718 non-null  object 
 14  Time_O        76696 non-null  object 
 15  Date_D        76718 non-null  object 
 16  Time_D        76688 non-null  object 
 17  Duration      76667 non-null  float64
 18  Purpose_O     76718 non-nu

### 3. Filter Individuals with GPS Records


In [16]:
print(f"Original number of individuals: {len(individuals_df)}")

# Filter for individuals with GPS records
individuals_gps_df = individuals_df[individuals_df['GPS_RECORD'] == True].copy()

print(f"Number of individuals with GPS records: {len(individuals_gps_df)}")

# Store the IDs of individuals with GPS data for filtering the trips table
gps_individual_ids = individuals_gps_df['ID'].unique()

Original number of individuals: 3337
Number of individuals with GPS records: 3320


### 4. Clean and Filter Trips Dataset


In [18]:
print(f"Original number of trips: {len(trips_df)}")

# 1. Filter trips to only include individuals with GPS records
trips_filtered_df = trips_df[trips_df['ID'].isin(gps_individual_ids)].copy()
print(f"Number of trips after filtering for GPS individuals: {len(trips_filtered_df)}")

# 2. Define invalid trip entries to remove
invalid_modes = ['No_Trip', 'No_Traces']

# 3. Remove rows with invalid modes
trips_cleaned_df = trips_filtered_df[~trips_filtered_df['Main_Mode'].isin(invalid_modes)]
print(f"Number of trips after removing '{', '.join(invalid_modes)}': {len(trips_cleaned_df)}")

# 4. Remove rows with missing Main_Mode
trips_cleaned_df.dropna(subset=['Main_Mode'], inplace=True)
print(f"Number of trips after removing missing Main_Mode: {len(trips_cleaned_df)}")

# Display final cleaned data
print("\n✅ Trips data cleaned and filtered.")
print("Final Cleaned Trips Dataset Head:")
display(trips_cleaned_df.head())

Original number of trips: 80697
Number of trips after filtering for GPS individuals: 80398
Number of trips after removing 'No_Trip, No_Traces': 80398
Number of trips after removing missing Main_Mode: 76463

✅ Trips data cleaned and filtered.
Final Cleaned Trips Dataset Head:


,KEY,ID,Day_EMG,Date_EMG,Day_Type,ID_Trip_Days,Type_Trip_OD,Area_O,Area_D,Code_INSEE_O,...,Duration,Purpose_O,Purpose_D,Main_Mode,Mode_1,Mode_2,Mode_3,Mode_4,Mode_5,Weight_Day
1,42_0001-tuesday-44852-1,42_0001,tuesday,2022-10-18,Strike,1,Internal,SAVIGNY-SUR-ORGE,SAINT-MAUR-DES-FOSSES,91589,...,72.0,DEPART_HOME,OTHER,PRIV_CAR_PASSENGER,PRIV_CAR_PASSENGER,NaN,NaN,NaN,NaN,92.090337
2,42_0001-tuesday-44852-2,42_0001,tuesday,2022-10-18,Strike,2,Internal,SAINT-MAUR-DES-FOSSES,SAINTE-GENEVIEVE-DES-BOIS,94068,...,39.0,OTHER,LEISURE,PRIV_CAR_PASSENGER,PRIV_CAR_PASSENGER,NaN,NaN,NaN,NaN,92.090337
3,42_0001-tuesday-44852-3,42_0001,tuesday,2022-10-18,Strike,3,Internal,SAINTE-GENEVIEVE-DES-BOIS,EPINAY-SUR-ORGE,91549,...,17.0,LEISURE,PURCHASE,PRIV_CAR_PASSENGER,PRIV_CAR_PASSENGER,NaN,NaN,NaN,NaN,92.090337
4,42_0001-tuesday-44852-4,42_0001,tuesday,2022-10-18,Strike,4,Internal,EPINAY-SUR-ORGE,SAVIGNY-SUR-ORGE,91216,...,6.0,PURCHASE,RETURN_HOME,PRIV_CAR_PASSENGER,PRIV_CAR_PASSENGER,NaN,NaN,NaN,NaN,92.090337
5,42_0001-wednesday-44853-1,42_0001,wednesday,2022-10-19,Normal,1,Internal,SAVIGNY-SUR-ORGE,SAVIGNY-SUR-ORGE,91589,...,5.0,DEPART_HOME,PURCHASE,PRIV_CAR_PASSENGER,PRIV_CAR_PASSENGER,NaN,NaN,NaN,NaN,89.587128


---